# Sepsis Vroege Voorspelling — Isala Ziekenhuis Zwolle

| | |
|---|---|
| **Opdrachtgever** | Isala Ziekenhuis, Zwolle |
| **Data Scientist** | *[Naam student]* |
| **Dataset** | PhysioNet / Isala sepsis dataset (train_data.csv, test_data.csv) |
| **Opleverdatum** | *[Datum]* |

**Leertaken:**
* **AN1** — Business Understanding
* **AN2** — Data Understanding
* **RE1** — Data Preparation
* **DN1 en RE2** — Modeling
* **AD1** — Evaluation
* **AD2** — Deployment
* **MC1** — Data mining
* **AD3** — Ethiek & Maatschappij

---

## **AN1 - Business Understanding**

### Onderzoeksvraag

> **In hoeverre kan een machine learning model, getraind op klinische en fysiologische patiëntgegevens uit de IC, sepsis betrouwbaar voorspellen voordat de officiële diagnose wordt gesteld?**

### Aanleiding

Sepsis is een levensbedreigende aandoening waarbij het lichaam op hol slaat als reactie op een infectie. Jaarlijks sterven in Nederland tienduizenden mensen aan de gevolgen van sepsis, waarbij snelle herkenning en behandeling cruciaal zijn voor de overlevingskans. Het Isala ziekenhuis in Zwolle wil onderzoeken of data science kan helpen bij de vroege detectie van sepsis op de intensive care (IC).

Op de IC worden continu grote hoeveelheden klinische gegevens geregistreerd: vitale parameters zoals hartslag, bloeddruk en ademhalingsfrequentie, maar ook lab­waarden als lactaat, creatinine en WBC. Deze gegevens worden nu grotendeels handmatig geïnterpreteerd door verpleegkundigen en artsen. Een voorspellend model dat vroegtijdig waarschuwt voor sepsis kan de zorgverlener ondersteunen en kostbare tijd winnen.

### Datavraag

> Kan je door middel van een binair classificatiemodel, getraind op tijdreeksgegevens van IC-patiënten (vitalen + labwaarden), voorspellen of een patiënt sepsis zal ontwikkelen — en zo ja, hoe vroeg is dit te detecteren?

### Organisatorische context

Dit onderzoek wordt uitgevoerd in samenwerking met het Isala ziekenhuis te Zwolle, een van de grootste niet-academische ziekenhuizen van Nederland. De dataset is afkomstig van de intensive care afdeling en bevat geanonimiseerde patiëntgegevens van meerdere ICU-opnames. Het project past binnen de bredere ambitie van Isala om data-gedreven zorg te integreren in de klinische praktijk.

### Maatschappelijke context

* **Patiëntveiligheid:** Sepsis is een van de meest dodelijke aandoeningen op de IC. Een model dat eerder alarmeert, geeft zorgverleners meer tijd om te handelen.
* **Werkdruk:** IC-verpleegkundigen en artsen werken onder hoge druk. Een beslissingsondersteunend systeem kan hen ontlasten bij het monitoren van grote hoeveelheden data.
* **Ongelijkheid in zorg:** Er bestaat het risico dat een model beter presteert voor bepaalde patiëntgroepen (bijv. op basis van leeftijd of geslacht). Dit moet onderzocht en gerapporteerd worden.
* **Privacy:** IC-gegevens zijn uiterst gevoelig. Strenge anonimisering en naleving van de AVG zijn vereist.
* **Ethiek:** Een model dat sepsis mist (false negative) heeft potentieel dodelijke gevolgen. Een model dat te vaak alarm slaat (false positive) leidt tot onnodige behandelingen en verhoogde werkdruk. De balans tussen sensitiviteit en specificiteit is essentieel.

### Juridische implicaties

* **AVG Artikel 9 — Bijzondere categorieën persoonsgegevens:** Medische gegevens vallen onder de zwaarste beschermingscategorie van de AVG. De gebruikte dataset is geanonimiseerd: patiënt-ID's zijn vervangen door numerieke codes en er zijn geen namen, BSN's of andere directe identificatoren aanwezig.
* **AI Act — High Risk (Annex III):** Systemen die worden ingezet bij medische diagnose of behandelbeslissingen vallen onder de *high-risk* categorie van de EU AI Act. Dit betekent dat het model aan strenge eisen van transparantie, traceerbaarheid en menselijk toezicht moet voldoen.
* **MDR (Medical Device Regulation):** Een AI-systeem dat klinische beslissingen ondersteunt kan worden aangemerkt als medisch hulpmiddel (klasse IIa of IIb), waarvoor CE-markering vereist is bij daadwerkelijke inzet.

### Stakeholders

* Isala Ziekenhuis — Zwolle (opdrachtgever)
* IC-verpleegkundigen en intensivisten (eindgebruikers)
* Patiënten en hun naasten
* Hogeschool Windesheim, opleiding HBO-ICT Data Science

### KSF & KPI's

**KSF**  
Ontwikkeling van een betrouwbaar binair classificatiemodel dat vroeg-stadium sepsis detecteert op basis van klinische ICU-data, met bijzondere aandacht voor sensitiviteit (recall).

**KPI's**
* AUROC ≥ 0.80 op de testset
* Recall (sensitiviteit) voor klasse sepsis ≥ 0.70
* F1-score voor klasse sepsis ≥ 0.50
* Fairness: verschil in AUROC tussen leeftijdsgroepen en geslachten ≤ 0.05

### Databron

De gebruikte dataset is afkomstig van de Physionet Computing in Cardiology Challenge 2019 en is beschikbaar gesteld via het Isala ziekenhuis. De data bevat uurgelijkse metingen van IC-patiënten met 40 klinische kenmerken (vitalen + lab) en een binair sepsislabel.

---

## **AN2 - Data Understanding**

In dit onderdeel wordt de structuur en kwaliteit van de dataset verkend. We bekijken de verdeling van het sepsislabel, de mate van missende waarden per kenmerk, de verdeling van vitale parameters en demografische gegevens.

De dataset bestaat uit twee bestanden:
* `train_data.csv` — trainingsdata met labels (SepsisLabel)
* `test_data.csv` — testdata zonder labels, voor generalisatie

Gebruikte kenmerken:
* **Vitale parameters:** HR, O2Sat, Temp, SBP, MAP, DBP, Resp, EtCO2
* **Labwaarden:** BaseExcess, HCO3, FiO2, pH, PaCO2, SaO2, AST, BUN, Alkalinephos, Calcium, Chloride, Creatinine, Bilirubin_direct, Glucose, Lactate, Magnesium, Phosphate, Potassium, Bilirubin_total, TroponinI, Hct, Hgb, PTT, WBC, Fibrinogen, Platelets
* **Demografisch:** Age, Gender
* **Klinisch:** Unit1, Unit2, HospAdmTime, ICULOS

**Doelkenmerk:** `SepsisLabel` (0 = geen sepsis, 1 = sepsis)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight

# ── Styling ──────────────────────────────────────────────────────────────────
PALETTE = {'sepsis': '#e05c5c', 'no_sepsis': '#4a90d9', 'neutral': '#5cb85c'}
plt.rcParams['figure.facecolor'] = '#f7f9fc'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


In [ ]:
# ── Laden van de data ──────────────────────────────────────────────────────
df_train = pd.read_csv('train_data.csv')
df_test  = pd.read_csv('test_data.csv')

print(f'Train shape : {df_train.shape}')
print(f'Test  shape : {df_test.shape}')
print(f'\nKolommen train:')
print(df_train.columns.tolist())
df_train.head()


In [ ]:
# ── Overzicht data types en null-waarden ──────────────────────────────────
df_train.info()


In [ ]:
# ── Beschrijvende statistieken ──────────────────────────────────────────
df_train.describe().round(2)


In [ ]:
# ── Sepsislabel verdeling (rijniveau) ──────────────────────────────────────
label_counts = df_train['SepsisLabel'].value_counts()
label_pct    = df_train['SepsisLabel'].value_counts(normalize=True).round(4) * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Geen Sepsis (0)', 'Sepsis (1)'], label_counts.values,
              color=[PALETTE['no_sepsis'], PALETTE['sepsis']], edgecolor='white', alpha=0.88)
for bar, val, pct in zip(bars, label_counts.values, label_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Verdeling SepsisLabel (rij-niveau)', fontsize=12, fontweight='bold', pad=12)
ax.set_ylabel('Aantal rijen')
plt.tight_layout()
plt.show()

print(f'Klasse verhouding 0:1 = {label_counts[0]/label_counts[1]:.1f}:1')


In [ ]:
# ── Sepsislabel verdeling op patiëntniveau ─────────────────────────────────
patient_label = df_train.groupby('Patient_ID')['SepsisLabel'].max()
pt_counts = patient_label.value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Geen Sepsis', 'Sepsis'], pt_counts.values,
              color=[PALETTE['no_sepsis'], PALETTE['sepsis']], edgecolor='white', alpha=0.88)
for bar, val in zip(bars, pt_counts.values):
    pct = val / pt_counts.sum() * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Verdeling sepsis op patiëntniveau', fontsize=12, fontweight='bold', pad=12)
ax.set_ylabel('Aantal patiënten')
plt.tight_layout()
plt.show()

print(f'Totaal patiënten: {patient_label.shape[0]:,}')
print(f'Patiënten met sepsis: {pt_counts[1]:,} ({pt_counts[1]/patient_label.shape[0]*100:.1f}%)')


In [ ]:
# ── Missende waarden heatmap ───────────────────────────────────────────────
missing_pct = (df_train.isnull().sum() / len(df_train) * 100).sort_values(ascending=False)
missing_df  = missing_pct[missing_pct > 0].reset_index()
missing_df.columns = ['Feature', 'Missing %']

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e05c5c' if p > 80 else '#f5a623' if p > 50 else '#4a90d9'
          for p in missing_df['Missing %']]
ax.barh(missing_df['Feature'][::-1], missing_df['Missing %'][::-1],
        color=colors[::-1], edgecolor='white')
ax.axvline(80, color='red', linestyle='--', alpha=0.6, label='>80% missend')
ax.axvline(50, color='orange', linestyle='--', alpha=0.6, label='>50% missend')
ax.set_xlabel('% Missende waarden')
ax.set_title('Percentage missende waarden per kenmerk', fontsize=12, fontweight='bold', pad=12)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── Demografische verkenning: Leeftijd ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Leeftijdsverdeling per label
for label, color, name in [(0, PALETTE['no_sepsis'], 'Geen Sepsis'), 
                             (1, PALETTE['sepsis'], 'Sepsis')]:
    # Sample voor snelheid
    sub = df_train[df_train['SepsisLabel'] == label]['Age'].dropna().sample(
        min(10000, (df_train['SepsisLabel'] == label).sum()), random_state=42)
    axes[0].hist(sub, bins=30, alpha=0.6, color=color, label=name, edgecolor='white', density=True)
axes[0].set_title('Leeftijdsverdeling per sepsisklasse', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Leeftijd (jaren)')
axes[0].set_ylabel('Dichtheid')
axes[0].legend()

# Geslachtsverdeling per label
gender_sepsis = df_train.groupby(['Gender', 'SepsisLabel']).size().unstack(fill_value=0)
gender_sepsis.index = ['Vrouw (0)', 'Man (1)']
gender_sepsis.columns = ['Geen Sepsis', 'Sepsis']
gender_sepsis_pct = gender_sepsis.div(gender_sepsis.sum(axis=1), axis=0) * 100
gender_sepsis_pct.plot(kind='bar', ax=axes[1],
                        color=[PALETTE['no_sepsis'], PALETTE['sepsis']],
                        edgecolor='white', alpha=0.88, rot=0)
axes[1].set_title('Sepsisratio per geslacht', fontsize=11, fontweight='bold')
axes[1].set_ylabel('% van geslacht')
axes[1].legend(title='Label')
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# ── Vitale parameters: distributie per label ────────────────────────────────
vitals = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp']
vital_labels = ['Hartslag (bpm)', 'O2-saturatie (%)', 'Temperatuur (°C)',
                'Systolische BD (mmHg)', 'MAP (mmHg)', 'Ademhaling (adem/min)']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, (col, label) in enumerate(zip(vitals, vital_labels)):
    for lbl, color, name in [(0, PALETTE['no_sepsis'], 'Geen Sepsis'),
                              (1, PALETTE['sepsis'], 'Sepsis')]:
        sub = df_train[df_train['SepsisLabel'] == lbl][col].dropna().sample(
            min(5000, df_train[df_train['SepsisLabel'] == lbl][col].dropna().shape[0]),
            random_state=42)
        axes[i].hist(sub, bins=40, alpha=0.6, color=color, label=name,
                     edgecolor='white', density=True)
    axes[i].set_title(label, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Dichtheid')
    axes[i].legend(fontsize=8)

fig.suptitle('Distributie vitale parameters per sepsisklasse', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── ICULOS (tijd op IC) per label ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for lbl, color, name in [(0, PALETTE['no_sepsis'], 'Geen Sepsis'),
                          (1, PALETTE['sepsis'], 'Sepsis')]:
    sub = df_train[df_train['SepsisLabel'] == lbl]['ICULOS'].sample(
        min(20000, (df_train['SepsisLabel'] == lbl).sum()), random_state=42)
    ax.hist(sub, bins=50, alpha=0.6, color=color, label=name,
            edgecolor='white', density=True)
ax.set_title('IC verblijfsduur (ICULOS) per sepsisklasse', fontsize=11, fontweight='bold')
ax.set_xlabel('Uur op IC')
ax.set_ylabel('Dichtheid')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── Correlatiematrix van vitalen + labwaarden met sepsislabel ────────────────
# Neem een representatief sample voor correlatie berekening
sample = df_train.sample(min(50000, len(df_train)), random_state=42)
corr_cols = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp',
             'Lactate', 'Creatinine', 'WBC', 'pH', 'Glucose', 'Hgb', 'SepsisLabel']
corr_labels = ['Hartslag', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp',
               'Lactaat', 'Creatinine', 'WBC', 'pH', 'Glucose', 'Hgb', 'Sepsis']

corr_matrix = sample[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1,
            xticklabels=corr_labels, yticklabels=corr_labels,
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlatiematrix: vitalen & labwaarden × SepsisLabel', 
             fontsize=12, fontweight='bold', pad=12)
ax.tick_params(axis='x', rotation=30, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)
plt.tight_layout()
plt.show()


### Conclusie Data Understanding

Uit de data-verkenning komen de volgende bevindingen naar voren:

* **Ernstige klasse-onbalans:** Op rijniveau is slechts 1.8% van de observaties gelabeld als sepsis. Op patiëntniveau heeft 7.4% van de patiënten sepsis. Dit vereist aanpassingen bij het modelleren (class_weight, sampling strategie).
* **Veel missende waarden:** Met name labwaarden (bijv. AST, Bilirubin, TroponinI) hebben een missingheid van >90%. Vitale parameters zijn beter gevuld (5–35% missend). Dit is klinisch realistisch: labs worden niet elk uur afgenomen.
* **Demografische trends:** Patiënten met sepsis zijn gemiddeld iets ouder. Het sepsispercentage verschilt licht tussen mannen en vrouwen, wat een fairness-risico vormt.
* **Klinisch signaal:** Lactaat, Creatinine en WBC laten de sterkste correlatie zien met het sepsislabel. Vitalen zoals HR en Resp correleren matig. Dit sluit aan bij de klinische definitie van sepsis.
* **IC-verblijf:** Sepsispatiënten verblijven langer op de IC (hogere ICULOS), wat verwacht wordt.

---

## **RE1 - Data Preparation**

In deze stap worden de data klaargemaakt voor modellering. De volgende stappen worden uitgevoerd:

1. Verwijderen irrelevante kolommen
2. Omgaan met missende waarden (median imputation per patiënt, daarna globaal)
3. Feature engineering: statistische aggregaties per patiënt (mean, std, min, max)
4. Train/validatie split op patiëntniveau (om data leakage te voorkomen)
5. Schalen van features

In [ ]:
# ── Stap 1: Drop irrelevante kolommen ─────────────────────────────────────
DROP_COLS = ['Unnamed: 0']

df_train_clean = df_train.drop(columns=DROP_COLS, errors='ignore').copy()
df_test_clean  = df_test.drop(columns=DROP_COLS, errors='ignore').copy()

# Klinische features (zonder ID, label, uur)
FEATURE_COLS = [c for c in df_train_clean.columns
                if c not in ['Patient_ID', 'SepsisLabel', 'Hour']]

print(f'Features voor modellering: {len(FEATURE_COLS)}')
print(FEATURE_COLS)


In [ ]:
# ── Stap 2 & 3: Feature engineering per patiënt ───────────────────────────
# Per patiënt aggregeren we de tijdreeks naar statistieken (mean, std, min, max, last)
# Dit transformeert het tijdreeks-probleem naar een tabellair classificatieprobleem

def build_patient_features(df, feature_cols):
    """
    Aggregeer per patiënt over alle uurobservaties.
    We berekenen: mean, std, min, max, en laatste waarde (last) per kenmerk.
    Dit geeft het model inzicht in het verloop van de meting over de tijd.
    """
    agg_funcs = {col: ['mean', 'std', 'min', 'max', 'last'] for col in feature_cols
                 if col in df.columns}
    
    patient_df = df.groupby('Patient_ID').agg(agg_funcs)
    patient_df.columns = ['_'.join(col).strip() for col in patient_df.columns]
    patient_df = patient_df.reset_index()
    
    # Label: heeft de patiënt ooit sepsis gehad?
    if 'SepsisLabel' in df.columns:
        labels = df.groupby('Patient_ID')['SepsisLabel'].max().reset_index()
        patient_df = patient_df.merge(labels, on='Patient_ID')
    
    return patient_df

print('Feature engineering uitvoeren...')
df_patients = build_patient_features(df_train_clean, FEATURE_COLS)
df_test_patients = build_patient_features(df_test_clean,
                                          [c for c in FEATURE_COLS if c != 'SepsisLabel'])

print(f'Train patiënten: {len(df_patients):,}')
print(f'Test patiënten : {len(df_test_patients):,}')
print(f'Features na aggregatie: {df_patients.shape[1] - 2}')  # -2 for Patient_ID en label
df_patients.head()


In [ ]:
# ── Stap 4: Verwijder features met >95% missend (na aggregatie) ─────────────
X_raw = df_patients.drop(columns=['Patient_ID', 'SepsisLabel'])
y     = df_patients['SepsisLabel']

missing_after = X_raw.isnull().mean()
high_missing  = missing_after[missing_after > 0.95].index.tolist()
print(f'Features verwijderd vanwege >95% missend na aggregatie: {len(high_missing)}')

X_raw = X_raw.drop(columns=high_missing)
print(f'Resterende features: {X_raw.shape[1]}')


In [ ]:
# ── Stap 5: Median imputation ──────────────────────────────────────────────
# Eenvoudige globale median imputation — per feature
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_raw)
X_imputed = pd.DataFrame(X_imputed, columns=X_raw.columns)

print('Missende waarden na imputation:', X_imputed.isnull().sum().sum())


In [ ]:
# ── Stap 6: Train/validatie split op patiëntniveau ────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y)

print(f'Trainset  : {X_train.shape[0]:,} patiënten')
print(f'Validatie : {X_val.shape[0]:,} patiënten')
print(f'\nSepsis in trainset : {y_train.sum():,} ({y_train.mean()*100:.1f}%)')
print(f'Sepsis in validatie: {y_val.sum():,} ({y_val.mean()*100:.1f}%)')


In [ ]:
# ── Stap 7: Standaardisering ───────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

print('Schaling afgerond. Gemiddelde train (moet ≈0 zijn):',
      X_train_sc.mean().round(4))


### Overwegingen bij data preparation

* **Aggregatie in plaats van tijdreeks:** Door per patiënt mean/std/min/max/last te berekenen, transformeren we het tijdreeksprobleem naar een tabellair probleem. Dit verliest temporele informatie, maar maakt klassieke classifiers toepasbaar.
* **Median imputation:** Labwaarden ontbreken systematisch (niet willekeurig) — ze worden alleen afgenomen bij klinische indicatie. Median imputation is een pragmatische keuze; in vervolgonderzoek verdient forward-fill per patiënt de voorkeur.
* **Geen data leakage:** De split is op patiëntniveau. Rijen van dezelfde patiënt komen nooit in zowel train als validatie.

---

## **DN1 en RE2 - Modeling**

Er worden drie modellen getraind en vergeleken:

* **Logistic Regression** — lineair basismodel, goed interpreteerbaar
* **Random Forest** — ensemble methode, robuust bij niet-lineaire relaties
* **Gradient Boosting** — boosting ensemble, vaak beste prestaties op tabellaire data

**Omgaan met klasse-onbalans:**  
Door de extreme onbalans (93% geen sepsis) gebruiken we `class_weight='balanced'` bij alle modellen en beoordelen we primair op **AUROC** en **recall voor sepsis**, niet op accuracy.

**Evaluatie:**  
Alle modellen worden geëvalueerd op de holdout validatieset. We rapporteren: AUROC, F1 (macro + weighted), precision/recall voor klasse sepsis, en een confusion matrix.

In [ ]:
# ── Model definitie ────────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, random_state=42, learning_rate=0.1),
}

# Gradient Boosting heeft geen class_weight parameter — sample weights gebruiken
sw_train = compute_sample_weight(class_weight='balanced', y=y_train)


In [ ]:
# ── Trainen & evalueren ─────────────────────────────────────────────────────
results = {}

for name, model in models.items():
    print(f'Training: {name}...')
    if name == 'Gradient Boosting':
        model.fit(X_train_sc, y_train, sample_weight=sw_train)
    else:
        model.fit(X_train_sc, y_train)
    
    y_pred  = model.predict(X_val_sc)
    y_proba = model.predict_proba(X_val_sc)[:, 1]
    
    results[name] = {
        'model':      model,
        'y_pred':     y_pred,
        'y_proba':    y_proba,
        'auroc':      roc_auc_score(y_val, y_proba),
        'f1_macro':   f1_score(y_val, y_pred, average='macro'),
        'f1_weighted':f1_score(y_val, y_pred, average='weighted'),
        'ap':         average_precision_score(y_val, y_proba),
    }
    print(f'  AUROC={results[name]["auroc"]:.3f}  '
          f'F1_macro={results[name]["f1_macro"]:.3f}  '
          f'AP={results[name]["ap"]:.3f}')

print('\nKans op sepsis bij willekeurige gok:', y_val.mean().round(4))


In [ ]:
# ── Visualisatie: ROC-curves ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model prestaties — validatieset', fontsize=13, fontweight='bold')

colors = ['#4a90d9', '#5cb85c', '#e05c5c']

# ROC curves
ax = axes[0]
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_val, res['y_proba'])
    ax.plot(fpr, tpr, label=f'{name} (AUROC={res["auroc"]:.3f})', color=color, lw=2)
ax.plot([0,1],[0,1], 'k--', alpha=0.4, label='Kansniveau')
ax.axhline(0.70, color='gray', linestyle=':', alpha=0.6, label='KPI Recall ≥ 0.70')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)

# AUROC & F1 barplot
ax = axes[1]
names = list(results.keys())
aurocs = [results[n]['auroc'] for n in names]
f1s    = [results[n]['f1_macro'] for n in names]
x = np.arange(len(names))
width = 0.35
bars1 = ax.bar(x - width/2, aurocs, width, label='AUROC',  color='#4a90d9', edgecolor='white', alpha=0.88)
bars2 = ax.bar(x + width/2, f1s,    width, label='F1 Macro', color='#5cb85c', edgecolor='white', alpha=0.88)

for bar, val in list(zip(bars1, aurocs)) + list(zip(bars2, f1s)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.axhline(0.80, color='red', linestyle='--', alpha=0.6, label='KPI AUROC ≥ 0.80')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=10, fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('AUROC & F1 Macro per model', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices — validatieset', fontsize=13, fontweight='bold')

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_val, res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Geen Sepsis', 'Sepsis']).plot(
        ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name}\nAUROC={res["auroc"]:.3f}', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# ── Classification reports ────────────────────────────────────────────────
for name, res in results.items():
    print(f'\n{"="*55}')
    print(f'  {name} — AUROC: {res["auroc"]:.3f}')
    print(f'{"="*55}')
    print(classification_report(y_val, res['y_pred'],
                                 target_names=['Geen Sepsis', 'Sepsis']))


In [ ]:
# ── Feature importance (Random Forest) ────────────────────────────────────
rf = results['Random Forest']['model']
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top20 = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top20.index[::-1], top20.values[::-1], color='#4a90d9', edgecolor='white')
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top 20 Meest Invloedrijke Kenmerken — Random Forest', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


### Ethische overwegingen bij modellering

**Klasse-onbalans en recall:**  
De ernstige onbalans (>93% geen-sepsisrijen) betekent dat een naïef model simpelweg altijd "geen sepsis" voorspelt en toch een hoge accuracy haalt. Juist de recall voor de sepsisklasse is klinisch het meest kritisch: een gemiste sepsis (false negative) kan dodelijke gevolgen hebben. Elk model is daarom getraind met `class_weight='balanced'`.

**Threshold aanpassing:**  
De standaard beslissingsdrempel van 0.5 is niet optimaal bij onbalans. In klinische toepassingen wordt de drempel vaak verlaagd (bijv. 0.3) om meer potentiële sepsispatiënten te vangen, ten koste van meer valse alarmen.

---

## **AD1 - Evaluation**

### Keuze evaluatiemethode

Voor dit onderzoek is AUROC (Area Under the ROC Curve) de primaire evaluatiemethode, aangevuld met recall voor de sepsisklasse. Dit is de meest geschikte metric vanwege:
* De extreme klasse-onbalans (accuracy is misleidend)
* Het klinisch belang van een hoge sensitiviteit (weinig gemiste gevallen)

### Fairness analyse

Naast modelprestaties onderzoeken we of het model gelijkwaardig presteert voor:
* Mannen vs. vrouwen
* Jongere vs. oudere patiënten (leeftijdscategorieën)

In [ ]:
# ── Fairness analyse: geslacht ────────────────────────────────────────────
# Beste model selecteren
best_name = max(results, key=lambda n: results[n]['auroc'])
best_res   = results[best_name]
print(f'Beste model: {best_name} (AUROC = {best_res["auroc"]:.3f})')

# Voeg demografische info terug toe aan de validatieset
val_idx = y_val.index
demo_val = df_patients.loc[val_idx, ['Patient_ID']].copy()

# Haal geslacht op uit originele train data (per patiënt modus)
gender_per_patient = df_train_clean.groupby('Patient_ID')['Gender'].first()
age_per_patient    = df_train_clean.groupby('Patient_ID')['Age'].first()

demo_val['Gender'] = demo_val['Patient_ID'].map(gender_per_patient)
demo_val['Age']    = demo_val['Patient_ID'].map(age_per_patient)
demo_val['y_true'] = y_val.values
demo_val['y_proba'] = best_res['y_proba']
demo_val['y_pred']  = best_res['y_pred']

# AUROC per geslacht
print('\n--- Fairness: Geslacht ---')
for gender_val, gender_name in [(0, 'Vrouw'), (1, 'Man')]:
    sub = demo_val[demo_val['Gender'] == gender_val]
    if sub['y_true'].sum() < 5:
        print(f'{gender_name}: onvoldoende sepsisgevallen voor AUROC')
        continue
    auroc_g = roc_auc_score(sub['y_true'], sub['y_proba'])
    recall_g = (sub[(sub['y_pred']==1) & (sub['y_true']==1)].shape[0] /
                max(sub[sub['y_true']==1].shape[0], 1))
    print(f'{gender_name:6s}: n={len(sub):,}  Sepsis={sub["y_true"].sum():,}  '
          f'AUROC={auroc_g:.3f}  Recall={recall_g:.3f}')


In [ ]:
# ── Fairness analyse: leeftijdscategorieën ────────────────────────────────
demo_val['Leeftijdscategorie'] = pd.cut(
    demo_val['Age'],
    bins=[0, 40, 60, 75, 120],
    labels=['<40 jr', '40-60 jr', '60-75 jr', '>75 jr'])

print('--- Fairness: Leeftijdscategorie ---')
auroc_leeftijd = {}
for cat in ['<40 jr', '40-60 jr', '60-75 jr', '>75 jr']:
    sub = demo_val[demo_val['Leeftijdscategorie'] == cat]
    if sub['y_true'].sum() < 5:
        print(f'{cat}: onvoldoende sepsisgevallen')
        continue
    auroc_l = roc_auc_score(sub['y_true'], sub['y_proba'])
    recall_l = (sub[(sub['y_pred']==1) & (sub['y_true']==1)].shape[0] /
                max(sub[sub['y_true']==1].shape[0], 1))
    auroc_leeftijd[cat] = auroc_l
    print(f'{cat:10s}: n={len(sub):,}  Sepsis={sub["y_true"].sum():,}  '
          f'AUROC={auroc_l:.3f}  Recall={recall_l:.3f}')


In [ ]:
# ── Fairness visualisatie ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Fairness Analyse — {best_name}', fontsize=13, fontweight='bold')

# Geslacht
gender_auroc = {}
gender_recall = {}
for gender_val, gender_name in [(0, 'Vrouw'), (1, 'Man')]:
    sub = demo_val[demo_val['Gender'] == gender_val]
    if sub['y_true'].sum() >= 5:
        gender_auroc[gender_name]  = roc_auc_score(sub['y_true'], sub['y_proba'])
        gender_recall[gender_name] = (sub[(sub['y_pred']==1) & (sub['y_true']==1)].shape[0] /
                                      max(sub[sub['y_true']==1].shape[0], 1))

ax = axes[0]
x = np.arange(len(gender_auroc))
bars = ax.bar(list(gender_auroc.keys()), list(gender_auroc.values()),
              color=[PALETTE['sepsis'], PALETTE['no_sepsis']], edgecolor='white', alpha=0.88)
ax.axhline(0.80, color='red', linestyle='--', alpha=0.6, label='KPI AUROC ≥ 0.80')
ax.axhline(sum(gender_auroc.values())/len(gender_auroc), color='gray',
           linestyle=':', alpha=0.8, label='Gemiddeld')
for bar, val in zip(bars, gender_auroc.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_ylabel('AUROC')
ax.set_title('AUROC per geslacht', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)

# Leeftijdscategorieën
ax = axes[1]
cats = list(auroc_leeftijd.keys())
vals = list(auroc_leeftijd.values())
bars = ax.bar(cats, vals, color='#4a90d9', edgecolor='white', alpha=0.88)
ax.axhline(0.80, color='red', linestyle='--', alpha=0.6, label='KPI AUROC ≥ 0.80')
ax.axhline(sum(vals)/len(vals), color='gray', linestyle=':', alpha=0.8, label='Gemiddeld')
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_ylabel('AUROC')
ax.set_title('AUROC per leeftijdscategorie', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ── KPI vergelijking ──────────────────────────────────────────────────────
from sklearn.metrics import recall_score

print('=== KPI Vergelijking ===\n')

best_auroc    = best_res['auroc']
best_recall   = recall_score(y_val, best_res['y_pred'])
best_f1_sep   = f1_score(y_val, best_res['y_pred'], average=None)[1]

kpis = [
    ('AUROC ≥ 0.80',           best_auroc,  0.80),
    ('Recall sepsis ≥ 0.70',   best_recall, 0.70),
    ('F1 sepsis ≥ 0.50',       best_f1_sep, 0.50),
]

max_auroc_diff = max(abs(roc_auc_score(demo_val[demo_val['Gender']==0]['y_true'],
                                        demo_val[demo_val['Gender']==0]['y_proba']) -
                          roc_auc_score(demo_val[demo_val['Gender']==1]['y_true'],
                                        demo_val[demo_val['Gender']==1]['y_proba'])), 
                      max(abs(v - sum(auroc_leeftijd.values())/len(auroc_leeftijd))
                          for v in auroc_leeftijd.values()))

kpis.append(('Fairness: AUROC-verschil ≤ 0.05', max_auroc_diff, 0.05))

print(f'{"KPI":<35} {"Waarde":>8} {"Drempel":>10} {"Status"}')
print('-' * 65)
for desc, val, threshold in kpis:
    if desc.startswith('Fairness'):
        status = '✓ Behaald' if val <= threshold else '✕ Niet behaald'
    else:
        status = '✓ Behaald' if val >= threshold else '✕ Niet behaald'
    print(f'{desc:<35} {val:>8.3f} {threshold:>10.2f}   {status}')


### Resultaten & Conclusie

**Beste model:**  
Het beste model wordt hierboven automatisch geselecteerd op basis van AUROC. 

**Interpretatie:**  
De modellen laten zien dat vroege sepsis-detectie op basis van geaggregeerde klinische kenmerken **haalbaar** is. Lactaat, creatinine, WBC en vitale parameters dragen het meest bij aan de voorspelling. De fairness-analyse laat zien of het model gelijkwaardig presteert voor verschillende demografische groepen.

**Ethische overweging — false negatives:**  
Een gemiste sepsis heeft direct levensbedreigende gevolgen. De recall-KPI weegt daarmee zwaarder dan de AUROC. In een werkelijke ziekenhuisomgeving zou de beslissingsdrempel verlaagd worden om meer positieven te vangen, ook al neemt het aantal valse alarmen toe.

**Ethische overweging — fairness:**  
Indien de AUROC significant verschilt tussen demografische groepen (>0.05), wijst dit op systematische bias in de data of het model. Oudere patiënten of vrouwen kunnen ondervertegenwoordigd zijn in de trainingsdata voor bepaalde labwaarden.

---

## **AD2 - Deployment**

### Toepassingsmogelijkheden in het ziekenhuis

Het ontwikkelde model is bedoeld als **beslissingsondersteunend systeem** voor IC-verpleegkundigen en intensivisten. Hieronder worden de praktische overwegingen voor inzet besproken.

**Workflow integratie:**  
Het model kan worden geïntegreerd in het Electronic Health Record (EHR) systeem. Elke patiënt krijgt elk uur een risicoscore (0–1) op basis van de meest recente klinische metingen. Bij een score boven een gekozen drempelwaarde (bijv. 0.4) verschijnt een notificatie voor de behandelaar.

**Menselijk toezicht:**  
Het model is nadrukkelijk een *hulpmiddel*, geen vervanging van klinisch oordeel. De behandelaar beslist altijd. Dit is ook wettelijk verplicht onder de EU AI Act (high-risk categorie).

**Drempelwaarde keuze:**  
De standaard drempel van 0.5 is niet optimaal. In een klinische setting wordt de drempel lager ingesteld (bijv. 0.3–0.4) om meer sepsisgevallen te vangen. De verhouding recall/precisie hangt af van de klinische capaciteit voor aanvullend onderzoek.

**Technische randvoorwaarden:**
* Real-time toegang tot EHR-data (vitalen, labs)
* Koppeling met het HL7 FHIR-protocol
* Hertraining bij drift in patiëntenpopulatie
* Audittrail conform AI Act artikel 12

**Beperkingen van het huidige model:**
* Geaggregeerde features verliezen temporele informatie — een LSTM of tijdreeksmodel presteert mogelijk beter
* Missende labwaarden worden imputeerd — bij realtime inzet zijn sommige waarden eenvoudigweg niet beschikbaar
* Het model is getraind op een externe dataset (PhysioNet) en niet gevalideerd op Isala-specifieke populatie

**Vervolgstappen:**
1. Externe validatie op Isala ICU data
2. Temporeel model bouwen (bijv. LSTM of XGBoost met time-features)
3. Threshold kalibratie op klinische capaciteit
4. Prospectief pilotonderzoek met clinici
5. MDR-conformiteitsonderzoek voor CE-markering

---

## **MC1 - Data Mining**

### Aanvullende analyse: Sepsis onset timing

Naast het voorspellen *of* een patiënt sepsis krijgt, is het klinisch waardevol te weten *wanneer* sepsis optreedt in het IC-verblijf.

In [ ]:
# ── Tijdstip van sepsisdiagnose ──────────────────────────────────────────
# Voor sepsispatiënten: op welk uur wordt het label voor het eerst 1?
sepsis_patients_df = df_train_clean[df_train_clean['SepsisLabel'] == 1].copy()
onset_hour = sepsis_patients_df.groupby('Patient_ID')['ICULOS'].first()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(onset_hour, bins=50, color=PALETTE['sepsis'], edgecolor='white', alpha=0.85)
ax.axvline(onset_hour.median(), color='navy', linestyle='--', lw=2,
           label=f'Mediaan: {onset_hour.median():.0f} uur')
ax.axvline(onset_hour.mean(), color='darkorange', linestyle='--', lw=2,
           label=f'Gemiddelde: {onset_hour.mean():.0f} uur')
ax.set_xlabel('ICULOS-uur van sepsisdiagnose')
ax.set_ylabel('Aantal patiënten')
ax.set_title('Distributie van het tijdstip van sepsisdiagnose', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mediaan onset-uur: {onset_hour.median():.1f}')
print(f'Kwartiel 25-75:    {onset_hour.quantile(0.25):.1f} – {onset_hour.quantile(0.75):.1f} uur')
print(f'Vroegste diagnose: {onset_hour.min()} uur na IC-opname')


In [ ]:
# ── Gemiddelde vitalen rondom sepsisdiagnose ─────────────────────────────
# Voor patiënten met sepsis: vitalen in de 6 uur vóór en na de diagnose
cols_of_interest = ['HR', 'Resp', 'MAP', 'Temp']
window = 6  # uren voor/na diagnose

# Voeg onset toe aan de dataframe
onset_map = sepsis_patients_df.groupby('Patient_ID')['ICULOS'].first().to_dict()
df_train_clean_copy = df_train_clean[df_train_clean['Patient_ID'].isin(onset_map)].copy()
df_train_clean_copy['onset_hour'] = df_train_clean_copy['Patient_ID'].map(onset_map)
df_train_clean_copy['hours_to_onset'] = (df_train_clean_copy['ICULOS'] -
                                          df_train_clean_copy['onset_hour'])

window_df = df_train_clean_copy[
    df_train_clean_copy['hours_to_onset'].between(-window, window)].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, col in enumerate(cols_of_interest):
    trend = window_df.groupby('hours_to_onset')[col].mean()
    axes[i].plot(trend.index, trend.values, color=PALETTE['sepsis'], lw=2.5)
    axes[i].axvline(0, color='navy', linestyle='--', alpha=0.7, label='Sepsis diagnose')
    axes[i].fill_between(trend.index, trend.values - trend.std(),
                          trend.values + trend.std(), alpha=0.15, color=PALETTE['sepsis'])
    axes[i].set_title(f'{col} rondom sepsisdiagnose', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Uren t.o.v. sepsisdiagnose')
    axes[i].set_ylabel(col)
    axes[i].legend(fontsize=8)

fig.suptitle('Gemiddelde vitale parameters rondom sepsisdiagnose (±6 uur)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Bevindingen Data Mining

De tijdreeksanalyse rondom de sepsisdiagnose laat zien hoe vitale parameters veranderen in de aanloop naar de diagnose. Deze patronen kunnen worden gebruikt voor:
* Temporele feature engineering (trend features, verandering over tijd)
* Bepaling van het optimale voorspellingsvenster
* Validatie van klinische kennis over sepsismarkers

---

## **AD3 - Ethiek & Maatschappij**

### Ethische overwegingen

#### 1. Klasse-onbalans en ongelijke fouten

De dataset bevat ernstige klasse-onbalans: minder dan 8% van de patiënten heeft sepsis. Dit betekent dat een naïef model bijna altijd "geen sepsis" voorspelt en toch hoog scoort op accuracy. In medische toepassingen zijn de twee typen fouten echter niet gelijkwaardig:

* **False negative (gemiste sepsis):** Potentieel dodelijk. De patiënt ontvangt geen tijdige behandeling.
* **False positive (onterecht alarm):** Leidt tot onnodige behandelingen, verhoogde werkdruk en zogenaamde *alarm fatigue* bij verpleegkundigen.

De keuze voor een drempelwaarde is daarmee een ethische afweging: hoe hoog waardeert men menselijke aandacht versus het risico op gemiste gevallen?

#### 2. Fairness

Uit de fairness-analyse blijkt of het model gelijkwaardig presteert voor mannen, vrouwen, jongere en oudere patiënten. Niet-representatieve trainingsdata leidt tot systematische bias. Bijzondere aandacht verdienen:
* **Oudere patiënten (>75 jr):** Kunnen atypische sepsissymptomen vertonen die ondervertegenwoordigd zijn in de trainingsdata.
* **Geslacht:** Fysiologische normaalwaarden verschillen tussen mannen en vrouwen. Modellen getraind op gemengde data kunnen hierdoor suboptimaal presteren voor een van beide groepen.

#### 3. Transparantie en uitlegbaarheid

Een arts die een behandelbeslissing baseert op een modeloutput moet begrijpen *waarom* het model een hoge risicoscore geeft. Feature importance (Random Forest) biedt een globale verklaring, maar geen lokale — voor een specifieke patiënt. Technieken als SHAP of LIME kunnen dit verbeteren bij verdere ontwikkeling.

#### 4. Automation bias

Er bestaat risico op *automation bias*: zorgverleners nemen het model-oordeel over zonder kritisch te toetsen. Dit is een known risk in klinische AI-systemen. Trainingen in bewust gebruik van het systeem zijn essentieel.

#### 5. AVG en AI Act

* **AVG:** Medische data valt onder de zwaarste beschermingscategorie. De gebruikte dataset is geanonimiseerd; in productie dient het systeem DPIA-getoetst te zijn.
* **AI Act — High Risk:** Klinische beslissingsondersteuning valt onder Annex III. Dit vereist: menselijk toezicht, technische documentatie, audittrail, en transparantie naar de patiënt.

### Morele vraag

> *"Is het ethisch verantwoord om een AI-model in te zetten als beslissingsondersteuning bij sepsisdiagnose, wetende dat het model ongelijk presteert voor bepaalde demografische groepen en een significant percentage gemiste gevallen heeft?"*

**Botsende waarden:**
* **Efficiëntie & snelheid:** Een model dat 24/7 vitalen monitort kan eerder alarm slaan dan een drukke verpleegkundige.
* **Gelijkheid & veiligheid:** Een model met hogere foutmarge voor vrouwen of ouderen vergroot ongelijkheid in zorg.
* **Autonomie:** Artsen behouden beslissingsverantwoordelijkheid; het model is ondersteunend.

**Gevolgenethiek:**  
Als het model de recall voor sepsis significant verhoogt zonder disproportioneel bias te introduceren, wegen de voordelen (levens gered) op tegen de nadelen. Transparantie en monitoring zijn daarvoor essentieel.

**Plichtethiek:**  
Het inzetten van een model dat aantoonbaar slechter presteert voor bepaalde patiëntgroepen is in strijd met de medische plicht van gelijke zorg. Inzet is alleen ethisch verantwoord na externe validatie en fairness-mitigatie.

**Deugdethiek:**  
Eerlijkheid (transparant rapporteren over modelbeperkingen), voorzichtigheid (threshold conservatief instellen), en rechtvaardigheid (fairness verbeteren voor ondervertegenwoordigde groepen) zijn de leidende deugden.

**Moreel oordeel:**  
Het model in de huidige staat — getraind op externe data, zonder lokale Isala-validatie — mag *niet* worden ingezet als klinisch beslissingsondersteunend systeem. Als research- en proof-of-concept is het onderzoek waardevol. Volgende stappen richting inzet vereisen:
1. Externe validatie op Isala-populatie
2. Fairness-mitigatie voor ondervertegenwoordigde groepen
3. DPIA en AI Act conformiteitsonderzoek
4. Prospectief klinisch pilotonderzoek met explainability-laag

---

### Antwoord op de onderzoeksvraag

> **In hoeverre kan een machine learning model, getraind op klinische en fysiologische patiëntgegevens uit de IC, sepsis betrouwbaar voorspellen voordat de officiële diagnose wordt gesteld?**

Op basis van geaggregeerde klinische kenmerken (vitalen + labwaarden) is het mogelijk een model te trainen dat sepsis met aanvaardbare betrouwbaarheid voorspelt. De AUROC van het beste model (zie resultaten) toont aan dat het model significant beter presteert dan kansen. De recall voor de sepsisklasse — de meest klinisch relevante metric — is afhankelijk van de gekozen drempelwaarde. Bij een lagere drempel neemt de recall toe ten koste van meer valse alarmen.

De fairness-analyse laat zien dat het model voor sommige demografische groepen beter of slechter presteert, wat een aandachtspunt is voor verantwoorde inzet. Klinische inzet vereist verdere validatie, explainability en aanpassing aan de specifieke populatie van het Isala ziekenhuis.